# 3. Probabilidades out-of-fold com `cross_val_predict`

Este notebook aplica `cross_val_predict(..., cv=5, method="predict_proba")` ao XGBoost do projeto. Cada linha de `X_train` recebe uma probabilidade gerada por um modelo que não foi treinado naquela linha. Usamos essas probabilidades para avaliar o treino e escolher um threshold; o conjunto de teste fica reservado para a avaliação final.

O alvo segue a convenção do projeto: benigno = `0`, maligno = `1`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, fbeta_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import cross_val_predict
from breast_cancer.config import DATA_PATH, THRESHOLD_BETA
from breast_cancer.data import load_data
from breast_cancer.evaluation import evaluate_model
from breast_cancer.validation import select_threshold
from breast_cancer.models import make_xgboost
from breast_cancer.preprocessing import prepare_data

X_train, X_test, y_train, y_test = prepare_data(load_data(DATA_PATH))
print(f'Treino: {X_train.shape}; teste reservado: {X_test.shape}')
print(f'Proporção de malignos no treino: {y_train.mean():.1%}')

Treino: (341, 30); teste reservado: (228, 30)
Proporção de malignos no treino: 37.2%


## Gerar as previsões OOF

Com cinco folds, o sklearn ajusta cinco cópias do estimador. Em cada rodada, quatro partes treinam o modelo e a parte restante recebe probabilidades. `[:, 1]` seleciona a coluna da classe `1` (maligno).

In [2]:
model = make_xgboost()
y_proba_oof = cross_val_predict(
    model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba",
    n_jobs=1,
)[:, 1]

print(f'Probabilidades OOF: {y_proba_oof.shape[0]} (uma por observação de treino)')
print(f'Faixa: {y_proba_oof.min():.4f} a {y_proba_oof.max():.4f}')
print(f'AUC-ROC OOF: {roc_auc_score(y_train, y_proba_oof):.4f}')

Probabilidades OOF: 341 (uma por observação de treino)
Faixa: 0.0066 a 0.9930
AUC-ROC OOF: 0.9874


## Comparar com probabilidades in-sample

A comparação abaixo ajusta um modelo em todo o treino e calcula probabilidades nessas mesmas linhas. Essa AUC tende a ser mais otimista, pois o modelo já viu os exemplos. A OOF é uma estimativa mais honesta do desempenho em dados não vistos durante cada ajuste.

In [3]:
model.fit(X_train, y_train)
y_proba_in_sample = model.predict_proba(X_train)[:, 1]
comparison = pd.DataFrame(
    {
        'AUC-ROC': [
            roc_auc_score(y_train, y_proba_oof),
            roc_auc_score(y_train, y_proba_in_sample),
        ]
    },
    index=['OOF (validação cruzada)', 'In-sample (mesmos dados do ajuste)'],
)
comparison

,AUC-ROC
OOF (validação cruzada),0.987379
In-sample (mesmos dados do ajuste),0.998970


## Threshold escolhido nas probabilidades OOF

O projeto seleciona o threshold que maximiza F2 nas probabilidades OOF. Com β=2, o F-score dá peso maior ao recall, mantendo precision na média harmônica. O threshold é escolhido com `y_train`; em seguida, avaliamos no teste, que não participou da validação cruzada nem da escolha.

In [4]:
threshold_oof = select_threshold(
    y_train, y_proba_oof, beta=THRESHOLD_BETA
)
oof_metrics = evaluate_model(y_train, y_proba_oof, threshold_oof)
test_probabilities = model.predict_proba(X_test)[:, 1]
test_metrics = evaluate_model(y_test, test_probabilities, threshold_oof)

print(f'Threshold selecionado nas OOF: {threshold_oof:.4f}')
print(f'Precisão OOF para maligno: {precision_score(y_train, oof_metrics["predicted_labels"]):.3f}')
print(f'Recall OOF para maligno: {recall_score(y_train, oof_metrics["predicted_labels"]):.3f}')
print(f'F1 OOF para maligno: {f1_score(y_train, oof_metrics["predicted_labels"]):.3f}')
print(f'F2 OOF para maligno: {fbeta_score(y_train, oof_metrics["predicted_labels"], beta=THRESHOLD_BETA):.3f}')
print(f'AUC-ROC no teste: {test_metrics["auc_roc"]:.4f}')
print(f'Accuracy no teste: {test_metrics["accuracy"]:.4f}')
print('Relatório no teste (threshold OOF):')
print(test_metrics['classification_report'])
print('Matriz de confusão no teste (linhas reais, colunas previstas):\n', test_metrics['confusion_matrix'])

Threshold selecionado nas OOF: 0.4444
Precisão OOF para maligno: 0.945
Recall OOF para maligno: 0.953
F1 OOF para maligno: 0.949
F2 OOF para maligno: 0.951
AUC-ROC no teste: 0.9972
Accuracy no teste: 0.9737
Relatório no teste (threshold OOF):
              precision    recall  f1-score   support

     Benigno      0.966     0.993     0.979       143
     Maligno      0.988     0.941     0.964        85

    accuracy                          0.974       228
   macro avg      0.977     0.967     0.972       228
weighted avg      0.974     0.974     0.974       228

Matriz de confusão no teste (linhas reais, colunas previstas):
 [[142   1]
 [  5  80]]


## Como interpretar

- `y_proba_oof` contém uma probabilidade por observação de treino, sempre produzida por um fold que não treinou naquela observação.
- AUC OOF resume a capacidade de ordenar benignos e malignos sem depender de um threshold específico.
- O threshold é escolhido maximizando F2 usando rótulos e probabilidades OOF; por isso, precision/recall/F2 no próprio OOF podem ficar otimistas pela seleção. A AUC OOF não depende do threshold. Use o teste reservado para avaliar a regra completa e observe que outro split pode gerar outro threshold.
- Este é um experimento de avaliação; o notebook não altera o código de produção nem salva um novo artefato.